### Summary and Conclusion

This notebook brings together the key insights from the exploratory data analysis (EDA) of river water quality across South Africa and synthesizes the main spatial, distributional, and contextual patterns observed in the data. Building on the findings from the primary EDA notebooks, this section highlights overarching trends in water quality indicators, satellite-derived features, climatic variables, population density, and river system characteristics.

In addition to summarizing these patterns, this notebook focuses on examining relationships among variables through correlation analysis to identify potential associations between water quality measures and environmental or human-influenced factors. Collectively, these summaries and analyses distill the most salient findings, clarify how different drivers may interact, and support clear, data-driven conclusions within the project’s time constraints.

#### 1. Key Findings by Variable from EDA

##### a) Water Quality Indicators

Dissolved Reactive Phosphorus (DRP) exhibits a highly right-skewed distribution: most observations fall within generally acceptable ranges, while a small number of sites show substantially elevated concentrations. These high-DRP observations tend to cluster spatially rather than appearing randomly, suggesting localized nutrient enrichment driven by site-specific conditions such as downstream accumulation, urban activity, or agricultural runoff. River mouth and junction locations represent only a small fraction of sampling sites and unique geographic locations, indicating that while downstream effects are important, they are likely localized rather than dominant drivers of overall water quality variation in the dataset.

##### b) Satellite-Derived Spectral Features

Satellite-derived wavelength features (NIR, green, SWIR22) show unimodal but skewed distributions, with strong correlations among infrared bands—particularly between SWIR16 and SWIR22—indicating redundancy. Normalized indices (NDMI and MNDWI) behave differently, carrying complementary information related to vegetation moisture and surface water clarity. Spatially, higher NIR values tend to align with coastal and more urbanized regions, while NDMI patterns suggest higher moisture and vegetation presence in wetter coastal areas compared to the interior. These findings highlight both the usefulness and the multicollinearity risks of spectral features for downstream modeling.

##### c) Population Density

Population density exhibits extremely high variability across sampling locations, with the majority of sites situated in sparsely populated or rural areas and a small subset located in dense urban regions. These urban sites disproportionately influence the upper tail of the distribution, even after applying a log transformation. Spatially, higher population density values align with known urban centers, providing confidence in the population density integration. Overall, population density appears to function more as a contextual modifier—shaping local environmental pressures—rather than as a direct linear driver of water quality or spectral signals.

##### d) Climatic Variables

Potential Evapotranspiration (PET) shows clear geographic gradients across provinces, reflecting regional climatic differences within South Africa. PET exhibits a negative relationship with longitude and varies systematically across space, underscoring the strong role of climate in shaping environmental conditions. These climatic variables provide important background context for interpreting observed patterns in both water quality measurements and satellite-derived spectral features.

#### 2. Correlation Analysis

##### a) Environment Setup and Library Imports

In [ ]:
%pip install folium
%pip install geopandas
%pip install contextily

In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from utils import primary_dataset
import folium
import seaborn as sns
import contextily as ctx
from matplotlib.colors import Normalize
from branca.colormap import linear

##### b) Pulling the merged dataset

In [3]:
water_quality_df = pd.read_csv('data/wq.csv')
water_quality_df.head()

,province,country,latitude,longitude,sample date,nir,green,swir16,swir22,ndmi,...,total alkalinity,electrical conductance,dissolved reactive phosphorus,month,sample_year,pop_density_nn,distance_km_to_pd_cell,river_mouthORjunction,river_mouth,river_junction
0,Mpumalanga,South Africa,-26.861111,28.884722,2011-01-03,17658.5,9550.0,13746.5,10574.0,0.124566,...,74.720,162.9,163.0,2011-01-31,2011,5.049022,0.251555,0,0,0
1,Gauteng,South Africa,-26.450000,28.085833,2011-01-03,15210.0,10720.0,17974.0,14201.0,-0.083293,...,89.254,573.0,80.0,2011-01-31,2011,23.239988,0.419537,0,0,0
2,Free State,South Africa,-27.671111,27.236944,2011-01-03,14887.0,10943.0,13522.0,11403.0,0.048048,...,82.000,203.6,101.0,2011-01-31,2011,687.465759,0.069958,0,0,0
3,Free State,South Africa,-27.356667,27.286389,2011-01-03,16828.5,9502.5,12665.5,9643.0,0.141147,...,56.100,145.1,151.0,2011-01-31,2011,6.092811,0.232396,0,0,0
4,Free State,South Africa,-27.010111,26.698083,2011-01-04,12433.5,10433.5,9579.5,8531.5,0.129651,...,82.200,289.8,192.0,2011-01-31,2011,77.849716,0.466183,0,0,0


##### c) Defining and Labeling Water Quality Indicators

Why do we need this?

To support downstream correlation and comparative analyses, we convert continuous water quality measurements into simple, interpretable binary indicators (coded as 1 = acceptable and 0 = not acceptable) based on commonly cited threshold values. This allows us to summarize overall water quality status at each sampling location and to examine how environmental, climatic, and human-context variables relate to “good” versus “not acceptable” water quality conditions.

In [8]:
# --- Define acceptable water quality thresholds ---
DRP_MAX = 100              # µg/L
TA_MIN, TA_MAX = 20, 200   # mg/L
EC_MAX = 700               # µS/cm

water_quality_labeled_df = water_quality_df.copy()

# Ensure numeric columns
cols_numeric = [
    "dissolved reactive phosphorus",
    "total alkalinity",
    "electrical conductance"
]
water_quality_labeled_df[cols_numeric] = (
    water_quality_labeled_df[cols_numeric]
    .apply(pd.to_numeric, errors="coerce")
)

# Individual binary flags (1 = acceptable, 0 = not acceptable)
water_quality_labeled_df["drp_ok"] = (
    water_quality_labeled_df["dissolved reactive phosphorus"] <= DRP_MAX
).astype(int)

water_quality_labeled_df["ta_ok"] = (
    water_quality_labeled_df["total alkalinity"].between(TA_MIN, TA_MAX)
).astype(int)

water_quality_labeled_df["ec_ok"] = (
    water_quality_labeled_df["electrical conductance"] <= EC_MAX
).astype(int)

# Overall water quality label
water_quality_labeled_df["good_water_quality"] = (
    (water_quality_labeled_df["drp_ok"] == 1) &
    (water_quality_labeled_df["ta_ok"] == 1) &
    (water_quality_labeled_df["ec_ok"] == 1)
).astype(int)

# Preview
water_quality_labeled_df.head()

,province,country,latitude,longitude,sample date,nir,green,swir16,swir22,ndmi,...,sample_year,pop_density_nn,distance_km_to_pd_cell,river_mouthORjunction,river_mouth,river_junction,drp_ok,ta_ok,ec_ok,good_water_quality
0,Mpumalanga,South Africa,-26.861111,28.884722,2011-01-03,17658.5,9550.0,13746.5,10574.0,0.124566,...,2011,5.049022,0.251555,0,0,0,0,1,1,0
1,Gauteng,South Africa,-26.450000,28.085833,2011-01-03,15210.0,10720.0,17974.0,14201.0,-0.083293,...,2011,23.239988,0.419537,0,0,0,1,1,1,1
2,Free State,South Africa,-27.671111,27.236944,2011-01-03,14887.0,10943.0,13522.0,11403.0,0.048048,...,2011,687.465759,0.069958,0,0,0,0,1,1,0
3,Free State,South Africa,-27.356667,27.286389,2011-01-03,16828.5,9502.5,12665.5,9643.0,0.141147,...,2011,6.092811,0.232396,0,0,0,0,1,1,0
4,Free State,South Africa,-27.010111,26.698083,2011-01-04,12433.5,10433.5,9579.5,8531.5,0.129651,...,2011,77.849716,0.466183,0,0,0,0,1,1,0


##### d) Counts and Proportions of Water Quality Classes

In [10]:
# Count good (1) vs bad (0) water quality
quality_counts = water_quality_labeled_df["good_water_quality"].value_counts().sort_index()

# Convert to proportions
quality_proportions = (
    water_quality_labeled_df["good_water_quality"]
    .value_counts(normalize=True)
    .sort_index()
)

# Combine into a summary table
water_quality_summary = pd.DataFrame({
    "count": quality_counts,
    "proportion": quality_proportions
})

water_quality_summary

,count,proportion
good_water_quality,,
0,4053,0.445727
1,5040,0.554273


Observation: Just over half of the observations meet all defined water quality thresholds, resulting in a fairly balanced distribution between acceptable and non-acceptable water quality classes.